Ingest volume for storing lab files

In [0]:
 %sql
 CREATE VOLUME IF NOT EXISTS spark_lab


Download files from github

In [0]:
 import requests

 # Define the current catalog
 catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]

 # Define the base path using the current catalog
 volume_base = f"/Volumes/{catalog_name}/default/spark_lab"

 # List of files to download
 files = ["2019.csv", "2020.csv", "2021.csv"]

 # Download each file
 for file in files:
     url = f"https://raw.githubusercontent.com/MicrosoftLearning/mslearn-databricks/main/data/{file}"
     response = requests.get(url)
     response.raise_for_status()

     # Write to Unity Catalog volume
     with open(f"{volume_base}/{file}", "wb") as f:
         f.write(response.content)

Define the schema for the data

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
orderSchema = StructType([
     StructField("SalesOrderNumber", StringType()),
     StructField("SalesOrderLineNumber", IntegerType()),
     StructField("OrderDate", DateType()),
     StructField("CustomerName", StringType()),
     StructField("Email", StringType()),
     StructField("Item", StringType()),
     StructField("Quantity", IntegerType()),
     StructField("UnitPrice", FloatType()),
     StructField("Tax", FloatType())
])
df = spark.read.load(f'/Volumes/{catalog_name}/default/spark_lab/*.csv', format='csv', schema=orderSchema)
display(df.limit(100))

Clean the data and remove all the null values

In [0]:
 from pyspark.sql.functions import col
 df = df.dropDuplicates()
 df = df.withColumn('Tax', col('UnitPrice') * 0.08)
 df = df.withColumn('Tax', col('Tax').cast("float"))
 display(df.limit(100))

Changing the data type of the tax column back to float because it is changed to double after the calculation and double uses more memory

Next we filter the columns of the sales order dataframe to contain only the customer name and email. Then count the number of customers and display the distinct customers

In [0]:
customers = df['CustomerName', 'Email']
print(customers.count())
print(customers.distinct().count())
display(customers.distinct())

In [0]:
customers = df.select("CustomerName", "Email").where(df['Item']=='Road-250 Red, 52')
print(customers.count())
print(customers.distinct().count())
display(customers.distinct())

For only customers who have placed a specific order. Next we can aggregate and group the data. The groupBy method groups the rows by Item, and the subsequent sum aggregate function is applied to all of the remaining numeric columns (in this case, Quantity)

In [0]:
productSales = df.select("Item", "Quantity").groupBy("Item").sum()
display(productSales)

In [0]:
yearlySales = df.select(year("OrderDate").alias("Year")).groupBy("Year").count().orderBy("Year")
display(yearlySales)

This shows the number of sales per year. Note that the select method includes a SQL year function to extract the year component of the OrderDate field, and then an alias method is used to assign a column name to the extracted year value. The data is then grouped by the derived Year column and the count of rows in each group is calculated before finally the orderBy method is used to sort the resulting dataframe. 

In [0]:
df.createOrReplaceTempView("salesorders")

This code line will create a temporary view that can then be used directly with SQL statements.

In [0]:
%sql
    
SELECT YEAR(OrderDate) AS OrderYear,
       SUM((UnitPrice * Quantity) + Tax) AS GrossRevenue
FROM salesorders
GROUP BY YEAR(OrderDate)
ORDER BY OrderYear;